# 04_baseline_financial.ipynb

## XGBoost + SHAP Baseline for Financial (Stateful) Experiment

This is Experiment 2 from the thesis - the **STATEFUL** environment.

Key difference from grid:
- Target has **memory** (path dependency, autocorrelation)
- Predicting next-day return direction (UP/DOWN)
- This stress-tests whether LLMIP works in stateful environments

Target: `target_direction` (0=DOWN, 1=UP)

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.metrics import accuracy_score, confusion_matrix

print("Libraries loaded!")

/home/l1zle/LLMIP/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded!


## 2. Load Data

In [2]:
PROJECT_DIR = Path("/home/l1zle/LLMIP")
FIN_DIR = PROJECT_DIR / "data" / "financial"
RESULTS_DIR = PROJECT_DIR / "results"

train = pd.read_csv(FIN_DIR / "sp500_train.csv")
test = pd.read_csv(FIN_DIR / "sp500_test.csv")

# Targets
train_targets = train['target_direction']
test_targets = test['target_direction']

# Features (exclude target columns)
exclude = ['target_direction', 'next_return', 'date']
feature_cols = [c for c in train.columns if c not in exclude]
train_features = train[feature_cols]
test_features = test[feature_cols]

print("=== Financial Data Loaded ===")
print(f"Train: {train_features.shape}")
print(f"Test:  {test_features.shape}")
print(f"\nTarget: next-day return direction (0=DOWN, 1=UP)")
print(f"\nTrain target distribution:")
print(train_targets.value_counts())

=== Financial Data Loaded ===
Train: (764, 23)
Test:  (192, 23)

Target: next-day return direction (0=DOWN, 1=UP)

Train target distribution:
target_direction
1    401
0    363
Name: count, dtype: int64


## 3. Feature Descriptions

In [3]:
FEATURE_DESCRIPTIONS = {
    "close": "Closing price (USD)",
    "open": "Opening price (USD)",
    "high": "Daily high price (USD)",
    "low": "Daily low price (USD)",
    "volume": "Trading volume (shares)",
    "return_1d": "1-day return (lagged, feature)",
    "return_5d": "5-day cumulative return (lagged)",
    "return_20d": "20-day cumulative return (lagged)",
    "sma_5": "5-day simple moving average",
    "sma_20": "20-day simple moving average",
    "sma_50": "50-day simple moving average",
    "volatility_5d": "5-day realized volatility",
    "volatility_20d": "20-day realized volatility",
    "momentum": "Momentum indicator",
    "rsi": "Relative Strength Index (0-100)",
    "macd": "MACD line",
    "macd_signal": "MACD signal line",
    "bb_mid": "Bollinger Band middle",
    "bb_upper": "Bollinger Band upper",
    "bb_lower": "Bollinger Band lower",
    "bb_position": "Bollinger Band position (0-1)",
    "dayofweek": "Day of week (0-6)",
    "month": "Month (1-12)",
}

print("\n=== Features ===")
for col in feature_cols:
    desc = FEATURE_DESCRIPTIONS.get(col, col)
    print(f"  {col:20s} - {desc}")


=== Features ===
  close                - Closing price (USD)
  open                 - Opening price (USD)
  high                 - Daily high price (USD)
  low                  - Daily low price (USD)
  volume               - Trading volume (shares)
  return_1d            - 1-day return (lagged, feature)
  return_5d            - 5-day cumulative return (lagged)
  return_20d           - 20-day cumulative return (lagged)
  sma_5                - 5-day simple moving average
  sma_20               - 20-day simple moving average
  sma_50               - 50-day simple moving average
  volatility_5d        - 5-day realized volatility
  volatility_20d       - 20-day realized volatility
  momentum             - Momentum indicator
  rsi                  - Relative Strength Index (0-100)
  macd                 - MACD line
  macd_signal          - MACD signal line
  bb_mid               - Bollinger Band middle
  bb_upper             - Bollinger Band upper
  bb_lower             - Bollinger Band 

## 4. Train XGBoost Classifier

In [6]:
print("\n=== Training XGBoost Classifier ===")

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
)

model.fit(train_features, train_targets)

print("Model trained!")


=== Training XGBoost Classifier ===
Model trained!


## 5. Predictions

In [7]:
predictions = model.predict(test_features)
pred_proba = model.predict_proba(test_features)

accuracy = accuracy_score(test_targets, predictions)

print("\n=== XGBoost Results ===")
print(f"Accuracy: {accuracy:.2%}")
print(f"Correct: {sum(predictions == test_targets)} / {len(test_targets)}")

print("\nConfusion Matrix:")
cm = confusion_matrix(test_targets, predictions)
print(f"  Predicted DOWN | Predicted UP")
print(f"Actual DOWN:  {cm[0,0]:4d}  |  {cm[0,1]:4d}")
print(f"Actual UP:   {cm[1,0]:4d}  |  {cm[1,1]:4d}")


=== XGBoost Results ===
Accuracy: 50.00%
Correct: 96 / 192

Confusion Matrix:
  Predicted DOWN | Predicted UP
Actual DOWN:    25  |    60
Actual UP:     36  |    71


## 6. SHAP Feature Importance

In [8]:
print("\n=== Computing SHAP Values ===")

explainer = shap.TreeExplainer(model)
n_shap = min(100, len(test_features))
shap_values = explainer.shap_values(test_features.head(n_shap))

# Use mean absolute SHAP
shap_abs = np.abs(shap_values)
feature_importance = shap_abs.mean(axis=0)
top_features = pd.Series(feature_importance, index=feature_cols).sort_values(ascending=False)

print("\nTop 10 Features by SHAP:")
for feat, val in top_features.head(10).items():
    print(f"  {feat:20s} {val:.4f}")


=== Computing SHAP Values ===

Top 10 Features by SHAP:
  volatility_20d       0.3810
  return_1d            0.3704
  rsi                  0.2983
  volatility_5d        0.2132
  bb_position          0.1932
  volume               0.1844
  return_5d            0.1362
  macd_signal          0.1306
  sma_5                0.1230
  return_20d           0.0872


## 7. Visualizations

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SHAP feature importance
ax1 = axes[0]
top_n = 15
shap_top = top_features.head(top_n)
ax1.barh(range(top_n), shap_top.values[::-1], color='#2c3e50')
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(shap_top.index[::-1], fontsize=9)
ax1.set_xlabel('Mean |SHAP|')
ax1.set_title('XGBoost Financial - Top Features by SHAP')

# Confusion matrix
ax2 = axes[1]
im = ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['DOWN (0)', 'UP (1)'])
ax2.set_yticklabels(['DOWN (0)', 'UP (1)'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title(f'Confusion Matrix\nAccuracy={accuracy:.2%}')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgboost_financial_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to: {RESULTS_DIR / 'xgboost_financial_baseline.png'}")


Figure saved to: /home/l1zle/LLMIP/results/xgboost_financial_baseline.png


/tmp/ipykernel_37129/956031826.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Save Results

In [14]:
results = {
    "domain": "financial",
    "model": "XGBClassifier",
    "n_train": int(len(train_features)),
    "n_test": int(len(test_features)),
    "n_features": int(len(feature_cols)),
    "target": "next-day return direction (0=DOWN, 1=UP)",
    "metrics": {
        "accuracy": float(accuracy),
        "correct": int(sum(predictions == test_targets)),
        "total": int(len(test_targets)),
        "confusion_matrix": cm.tolist(),
    },
    "shap": {
        "n_samples_used": n_shap,
        "top_features": top_features.head(20).to_dict(),
    },
}

with open(RESULTS_DIR / 'xgboost_financial_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {RESULTS_DIR / 'xgboost_financial_results.json'}")

Results saved to: /home/l1zle/LLMIP/results/xgboost_financial_results.json


## 9. Summary

In [15]:
print("\n" + "="*60)
print("XGBOOST FINANCIAL BASELINE COMPLETE")
print("="*60)
print(f"\nAccuracy: {accuracy:.2%}")
print(f"Correct predictions: {sum(predictions == test_targets)} / {len(test_targets)}")

print("\nTop 5 Features:")
for i, (feat, val) in enumerate(top_features.head(5).items()):
    print(f"  {i+1}. {feat:20s} ({val:.4f})")

print("\nNext: Run 05_llmip_financial.ipynb for LLMIP pipeline")


XGBOOST FINANCIAL BASELINE COMPLETE

Accuracy: 50.00%
Correct predictions: 96 / 192

Top 5 Features:
  1. volatility_20d       (0.3810)
  2. return_1d            (0.3704)
  3. rsi                  (0.2983)
  4. volatility_5d        (0.2132)
  5. bb_position          (0.1932)

Next: Run 05_llmip_financial.ipynb for LLMIP pipeline
